# Supervised Fine-Tuning (SFT) of SmolLM-360M on Amazon SageMaker Training Jobs

## Lab 3 - Local CPU Inference: Base vs Fine-Tuned

In this notebook, we download the LoRA adapter produced by our SageMaker Training job and run **local CPU inference** to compare the base model against the fine-tuned variant side-by-side.

This demonstrates the impact of SFT without requiring any GPU or endpoint deployment.

***

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade --quiet

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## 1. Find the Latest Training Job

In [ ]:
model_id = "HuggingFaceTB/SmolLM-360M"
job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft"

In [ ]:
def get_last_job_name(job_name_prefix):
    """Find the most recent completed training job matching the prefix."""
    search_response = sm_client.search(
        Resource="TrainingJob",
        SearchExpression={
            "Filters": [
                {"Name": "TrainingJobName", "Operator": "Contains", "Value": job_name_prefix},
                {"Name": "TrainingJobStatus", "Operator": "Equals", "Value": "Completed"},
            ]
        },
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )

    jobs = [
        r["TrainingJob"]["TrainingJobName"]
        for r in search_response["Results"]
        if r["TrainingJob"]["TrainingJobName"].startswith(job_name_prefix)
    ]

    if not jobs:
        raise ValueError(f"No completed training jobs found with prefix '{job_name_prefix}'")
    return jobs[0]

In [ ]:
job_name = get_last_job_name(job_prefix)
print(f"Latest training job: {job_name}")

# Get the model artifact S3 URI from the training job
job_description = sm_client.describe_training_job(TrainingJobName=job_name)
model_s3_uri = job_description["ModelArtifacts"]["S3ModelArtifacts"]
print(f"Model artifact: {model_s3_uri}")

***

## 2. Download and Extract the Fine-Tuned Adapter

In [ ]:
import os
import tarfile
from urllib.parse import urlparse

# Parse S3 URI
parsed = urlparse(model_s3_uri)
s3_bucket = parsed.netloc
s3_key = parsed.path.lstrip("/")

# Download and extract
local_tar = "./model.tar.gz"
adapter_dir = "./finetuned_adapter"

print(f"Downloading {s3_key}...")
s3_client.download_file(s3_bucket, s3_key, local_tar)

print("Extracting...")
os.makedirs(adapter_dir, exist_ok=True)
with tarfile.open(local_tar, "r:gz") as tar:
    tar.extractall(path=adapter_dir)

os.remove(local_tar)

print(f"\nExtracted files:")
for f in sorted(os.listdir(adapter_dir)):
    print(f"  {f}")

***

## 3. Load Base Model and Fine-Tuned Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float32, device_map="cpu"
)
base_model.eval()

print("Loading fine-tuned model (base + LoRA adapter)...")
# Load a separate base model instance for the adapter since PeftModel modifies in-place
ft_base = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float32, device_map="cpu"
)
finetuned_model = PeftModel.from_pretrained(ft_base, adapter_dir)
finetuned_model.eval()

print("\n✅ Both models loaded successfully on CPU!")

***

## 4. Compare Inference: Base vs Fine-Tuned

We'll use the same prompt format from our training data and compare outputs.

In [ ]:
def generate(model, prompt, max_new_tokens=150):
    """Generate text from a prompt using the given model."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
        )
    # Only decode the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def build_prompt(question):
    return f"### User:\n{question}\n\n### Assistant:\n"

In [ ]:
# Test questions — mix of seen-style and unseen questions
test_questions = [
    "What is a database?",
    "How do I make coffee?",
    "What is the meaning of life?",
]

for question in test_questions:
    prompt = build_prompt(question)

    print(f"{'='*70}")
    print(f"❓ Question: {question}")
    print(f"{'='*70}")

    base_response = generate(base_model, prompt)
    print(f"\n🔵 BASE MODEL:\n{base_response}")

    ft_response = generate(finetuned_model, prompt)
    print(f"\n🟢 FINE-TUNED MODEL:\n{ft_response}")
    print()

### Observations

You should notice:
- The **base model** produces generic text completions with no leprechaun personality
- The **fine-tuned model** responds like an Irish leprechaun — without any system prompt telling it to!

This is the power of Supervised Fine-Tuning — we baked the leprechaun behavior directly into the model weights. No prompt engineering needed at inference time.

***

## 5. Cleanup

In [ ]:
import shutil

if os.path.exists(adapter_dir):
    shutil.rmtree(adapter_dir)
    print(f"Removed {adapter_dir}")

print("✅ Cleanup complete!")